# Attention Is All You Need

Word embeddings alone give us a powerful way to represent words as vectors, capturing semantic similarity and relationships.  
However, embeddings **by themselves** are not enough to build a model that truly understands language in context, let's see why:

#### **The big leap the Contextual Meaning**

Consider the sentences:

- *I went to the river bank*  
- *I went to the central bank*

The word **bank** appears in both sentences, but its meaning is completely different.

A static embedding (like Word2Vec or GloVe) assigns **one vector per word**, so *bank* always has the same representation, regardless of context.

This leads to ambiguity that embeddings alone cannot resolve.

**Attention fixes this** by allowing the model to look at surrounding words and decide which meaning is relevant.  
In the first sentence, attention will focus on *river*.  
In the second, it will focus on *central*.  
using attnion

This dynamic, context‑dependent interpretation is the core of modern LLMs; a  model using only embeddings cannot learn how words relate across long distances or how to reorder them.

#### The Attention formula Explained

$$
\text{Attention}(Q, K, V)
= \text{softmax}\!\left( \frac{QK^{T}}{\sqrt{d_k}} \right) V
$$

**Linear Projections(Q,K,V):**

(Q,K,V) are called linear projection, they represent in theory:

- Q : Query, what orher words is the prediction especting?
- K : Key, how this word appear to others?
- V : Value, is the word carrying?

They are the result of moltiplicating the imput (a phrase, or a bach of pharases) for a matrix of weights, one for each query(Q,K,W).

**Formula explanation**
- QKT -> Here we're doing the dot product between the query and each word in the phrase measuring the similarity between each word and the desired words the query, in the end the we obtain a square matrix of scores that measure the similarity.
- Root of The size of k -> The result of "QKT" is divided by the square root of the key dimension.  
This scaling keeps the values stable and improves softmax behavior.
- The role of the softmax : takes the attention scores and trasforms them in probabilities that shows how much each word is important for the attention, for example "in the phrase i went to bank to get a loan"
the model might want to take 0.5 information from bank and 0.3 information from loan what the model looks to is determined by the linear ptojections Q, K and V.
- The moltiplication with the projection V the "sintesisis": We have obtained the information how much take from a single word from the softmax so now we're just highlighting the most important pieces of information in the phrase and shadowing everything else.


### Two observations about the imputs of the transformer:

The imput that attention gets is a 3d array where 2d vectors represent a words, so a list of vectors is a phrase and 3d vectors are a batch of phrases.

### Code Implementation:

In [2]:
import numpy as np

def softmax(x):
    x = x - np.max(x, axis=-1, keepdims=True)  # stability
    return np.exp(x) / np.sum(np.exp(x), axis=-1, keepdims=True)

def self_attention(X, d_k):
    """
    X: input sequence of shape (seq_len, d_model)
    d_k: dimension of keys/queries
    """
    seq_len, d_model = X.shape

    # Weight matrices (random for demo)
    W_Q = np.random.randn(d_model, d_k)
    W_K = np.random.randn(d_model, d_k)
    W_V = np.random.randn(d_model, d_k)

    # 1. Compute Q, K, V
    Q = X @ W_Q   # (seq_len, d_k)
    K = X @ W_K   # (seq_len, d_k)
    V = X @ W_V   # (seq_len, d_k)

    # 2. Compute attention scores
    scores = Q @ K.T / np.sqrt(d_k)  # (seq_len, seq_len)

    # 3. Softmax to get attention weights
    weights = softmax(scores)  # (seq_len, seq_len)

    # 4. Weighted sum of values
    output = weights @ V  # (seq_len, d_k)

    return output, weights

X_dummy = np.random.randn(4, 3)
output, weights = self_attention(X_dummy, 3)
print(output)


[[-1.98223622 -1.68344207  0.76231688]
 [ 3.54374142 -0.19504326 -1.08077895]
 [-1.94559417 -1.43870997  0.75485291]
 [-0.88156455  0.43357632  0.42793086]]


### Masked Self‑Attention

Masked self‑attention is a variant of self‑attention used during the training of autoregressive transformer models; its purpose is simple but absolutely essential: **prevent the model from “seeing the future.”**

When generating text one token at a time, the model should only attend to:

- the current token  
- all previous tokens  
- **never** any token that comes after it in the sequence  

If the model were allowed to look ahead, it would “cheat” during training — a phenomenon known as **data leakage**.  
Masked attention solves this by applying a **causal mask** to the attention score matrix before the softmax step.

### Formula

$$
\text{MaskedAttention}(Q, K, V)
= \text{softmax}\!\left( 
\frac{QK^{T}}{\sqrt{d_k}} + M
\right) V
$$


### How the Masking Works

Recall that attention scores are computed as:

$$
\text{scores} = QK^\top
$$

This produces an N * N matrix where each row corresponds to a query token and each column corresponds to a token.

For masking (left‑to‑right) generation, we apply a mask such that:

- entries corresponding to **future positions** are set to negative infinite 
- entries corresponding to **past or current positions** remain unchanged  
After this masking, we apply softmax row‑wise.

$$
\text{attention} = \text{softmax}(\text{masked\_scores})
$$
Because softmax of negative infinity the model assigns **zero probability** to all future tokens.
Masked self‑attention enforces a strict temporal direction:
This makes the transformer behave like a left‑to‑right language model, even though internally it uses parallel matrix operations.
For a sequence of length 4, the causal mask looks like:
$$
M =
\begin{bmatrix}
0 & -\infty & -\infty & -\infty \\
0 & 0 & -\infty & -\infty \\
0 & 0 & 0 & -\infty \\
0 & 0 & 0 & 0
\end{bmatrix}
$$
After softmax, this becomes a lower‑triangular attention pattern.

### Why use masking?

Masked self‑attention ensures that:

- the model learns to predict the next token using only past context  
- training and inference behave consistently  
- no information leaks from future tokens  
- the model develops a genuine generative ability rather than memorizing full sequences  

This is one of the core architectural decisions that makes transformer decoders so powerful for text generation.

In [18]:
import numpy as np

def softmax(x):
    # Subtract max for numerical stability
    x = x - np.max(x, axis=-1, keepdims=True)
    return np.exp(x) / np.sum(np.exp(x), axis=-1, keepdims=True)

def masked_self_attention(X, d_k):
    seq_len, d_model = X.shape

    # Random weights for demo
    W_Q = np.random.randn(d_model, d_k)
    W_K = np.random.randn(d_model, d_k)
    W_V = np.random.randn(d_model, d_k)

    # 1. Projections
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V

    # 2. Scores
    scores = Q @ K.T / np.sqrt(d_k)

    # --- UPDATED MASKING STRATEGY ---
    
    # Create a mask for the Upper Triangle (k=1 excludes diagonal)
    mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
    print(mask)

    # Apply negative infinity directly using boolean indexing
    scores[mask] = -1e9

    print(scores)

    # 3. Softmax (exp(-inf) becomes 0.0)
    weights = softmax(scores)

    # 4. Output
    output = weights @ V

    return output, weights


X_dummy = np.random.randn(4, 3)
output, weights = masked_self_attention(X_dummy, 3)
print(output)

[[False  True  True  True]
 [False False  True  True]
 [False False False  True]
 [False False False False]]
[[ 8.34523938e-01 -1.00000000e+09 -1.00000000e+09 -1.00000000e+09]
 [-1.02257967e+00 -5.15575392e-01 -1.00000000e+09 -1.00000000e+09]
 [-9.13528991e-02  1.97648754e+00  6.76470330e+00 -1.00000000e+09]
 [-5.14814540e-01  5.11694713e-01 -3.83277130e-01  1.63928721e-01]]
[[-0.46588945 -2.52796775  1.41873508]
 [ 0.86988165  0.27878917 -0.68265769]
 [ 4.54095254 -2.59472013 -1.24723346]
 [ 1.83357182 -0.03028683 -1.10128553]]


### Multi‑Head Attention

Multi‑head attention is an extension of self‑attention that allows the model to look at the input from **multiple representation subspaces simultaneously**.  
Instead of computing a single set of attention weights, the model computes several attention “heads,” each with its own learned projections.

The key idea is simple but powerful:  
**different heads learn to focus on different types of relationships.**

Some heads may track syntactic structure, others semantic similarity, others long‑range dependencies.  
By combining them, the transformer gains a richer and more expressive understanding of the sequence.

$$
\text{MultiHead}(Q, K, V)
= \text{Concat}(\text{head}_1, \dots, \text{head}_h)\, W_O
$$


### Why Multiple Heads?

A single attention head compresses all interactions into one similarity space.  
But language is multi‑dimensional:

- a word may relate to another syntactically  
- or semantically  
- or positionally  
- or through long‑range context  

A single head cannot capture all of these patterns at once.  
Multi‑head attention solves this by splitting the model dimension into several smaller subspaces and learning separate Q, K, V projections for each.

Each head has dimensionality:
$$
d_k = d_{\text{model}} / h
$$
Each head computes its own attention:
$$
\text{head}_i = \text{softmax}\left( \frac{Q_i K_i^\top}{\sqrt{d_k}} \right) V_i
$$
This produces **h different attention outputs**, each capturing different relationships.

### Combining the Heads

Once all heads are computed, they are concatenated:

$$
\text{concat}(\text{head}_1, \ldots, \text{head}_h)
$$

This restores the original dimensionality;  
finally, a learned output projection mixes the heads:

$$
\text{output} = \text{concat}(\text{heads}) \, W_O
$$
This allows the model to integrate the diverse information extracted by each head.

### Why Multi‑Head Is The Future

Multi‑head attention gives the transformer:

- **richer feature extraction** (each head learns a different pattern)  
- **better gradient flow** (smaller subspaces stabilize training)  
- **parallel reasoning** (heads operate independently)  
- **expressive power** far beyond a single attention mechanism  

This architectural choice is one of the main reasons transformers outperform earlier sequence models:  
they can attend to multiple aspects of the input **at the same time**, in parallel, with learned specialization.

### How heads work in the big picure

For a model with 4 heads, attention looks like:

- Head 1: focuses on local context  
- Head 2: tracks long‑range dependencies  
- Head 3: aligns syntactic structure  
- Head 4: captures semantic similarity  

Each head sees the same sequence, but through a different learned “lens.”  
The final output blends all these perspectives into a unified representation.

### Here we've an example of masked multi head self attention :

In [ ]:
import numpy as np

def softmax(x):
    # Standard stable softmax
    x = x - np.max(x, axis=-1, keepdims=True)
    return np.exp(x) / np.sum(np.exp(x), axis=-1, keepdims=True)

def masked_multi_head_attention(X, d_model, n_heads):
    """
    X: input sequence (seq_len, d_model)
    d_model: total dimension of the model (must be divisible by n_heads)
    n_heads: number of parallel attention heads
    """
    seq_len, _ = X.shape
    
    # Calculate the dimension of each head (depth)
    # e.g., if d_model=8 and n_heads=2, then depth=4
    depth = d_model // n_heads
    assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

    # --- 1. Weights Initialization ---
    # We project Q, K, V for all heads at once using one large matrix
    W_Q = np.random.randn(d_model, d_model)
    W_K = np.random.randn(d_model, d_model)
    W_V = np.random.randn(d_model, d_model)
    W_O = np.random.randn(d_model, d_model) # Output projection

    # --- 2. Linear Projections ---
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V

    # --- 3. Split Heads (Reshape & Transpose) ---
    # Current shape: (seq_len, d_model) -> (seq_len, n_heads, depth)
    # We transpose to: (n_heads, seq_len, depth) to perform batch matmul
    Q = Q.reshape(seq_len, n_heads, depth).transpose(1, 0, 2)
    K = K.reshape(seq_len, n_heads, depth).transpose(1, 0, 2)
    V = V.reshape(seq_len, n_heads, depth).transpose(1, 0, 2)

    # --- 4. Scaled Dot-Product Attention (with Mask) ---
    # Matmul: (n_heads, seq_len, depth) @ (n_heads, depth, seq_len)
    # Result: (n_heads, seq_len, seq_len)
    # Note: K.swapaxes allows transposing only the last two dimensions
    scores = (Q @ K.swapaxes(-1, -2)) / np.sqrt(depth)

    # Create Mask (Upper Triangular)
    # This broadcasts automatically across the 'n_heads' dimension
    mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
    
    # Apply Mask (Future tokens get -1e9)
    scores[:, mask] = -1e9

    # Softmax on the last dimension
    attn_weights = softmax(scores) # Shape: (n_heads, seq_len, seq_len)

    # Apply weights to Values
    # (n_heads, seq_len, seq_len) @ (n_heads, seq_len, depth)
    # Result: (n_heads, seq_len, depth)
    scaled_attention = attn_weights @ V

    # --- 5. Concatenation ---
    # Transpose back: (seq_len, n_heads, depth)
    scaled_attention = scaled_attention.transpose(1, 0, 2)
    
    # Flatten last two dims: (seq_len, n_heads * depth) -> (seq_len, d_model)
    concat_output = scaled_attention.reshape(seq_len, d_model)

    # --- 6. Final Linear Projection ---
    output = concat_output @ W_O

    return output, attn_weights


# Parameters
SEQ_LEN = 4
D_MODEL = 8   # Embedding vector size
N_HEADS = 2   # We split the 8 features into 2 heads of 4 features each
    
X_dummy = np.random.randn(SEQ_LEN, D_MODEL)
    
final_output, head_weights = masked_multi_head_attention(X_dummy, D_MODEL, N_HEADS)
    
print(f"Input Shape: {X_dummy.shape}")       # (4, 8)
print(f"Output Shape: {final_output.shape}") # (4, 8) - Same size, contextualized
    
print("\n--- Inspecting Head 1 vs Head 2 ---")
print("Head 1 Weights (First row):\n", np.round(head_weights[0, 0, :], 2))
print("Head 2 Weights (First row):\n", np.round(head_weights[1, 0, :], 2))


Input Shape: (4, 8)
Output Shape: (4, 8)

--- Inspecting Head 1 vs Head 2 ---
Head 1 Weights (First row):
 [1. 0. 0. 0.]
Head 2 Weights (First row):
 [1. 0. 0. 0.]
